# COSC-651 Week 6: Planning and Stateful Workflows

Supports course learning outcomes 1, 3, 4, 5, and 7.

An investigation may be interrupted while support is waiting for evidence or a human reviewer is checking a follow-up. You need to know whether LedgerLine Assist can resume without losing observations, refreshing spent budgets, or recording the same approved request twice.

Implement and test a bounded plan and its executor. Close and reopen the state store to demonstrate recovery. Bind human approval to the exact proposed payload before recording a simulated follow-up, and show that a retry leaves one record. The local action ledger records an approved request; the exercise does not pay a supplier or change the invoice importer. Preserve the plan comparison and the attributed peer contribution required by this week.

Read the weekly lab in Canvas for the assignment and case context.

## Experimental question

Can the application recover a bounded plan after interruption while preserving evidence, permissions, and action history?

Read the weekly lab and rubric before changing the starter. Fixture mode is a deterministic test double. Its results do not measure LLM capability.

## Environment and data

Use the installed course environment. Run the notebook from a clean kernel.

In [ ]:
from pathlib import Path
import json, sys
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / 'src/maintenance_course/data.py').is_file()), None)
assert REPO_ROOT is not None, 'Open this notebook from a COSC-651-Labs checkout.'
sys.path[:0] = [str(REPO_ROOT / 'src'), str(REPO_ROOT)]
from maintenance_course.data import LAB_ROOT, DATA, cases, case_by_id
from maintenance_course.experiments import run_case, save_runs, payload, gather
from maintenance_course.baselines import direct, fixed
from maintenance_course.models import FixtureModel
from maintenance_course.tools import ToolRegistry
sys.path.insert(0, str(LAB_ROOT))
MODE = "fixture"  # Change only after configuring the course-approved live access.
print({"mode": MODE, "lab_root": str(LAB_ROOT), "development_cases": len(cases())})

## Provided baseline

Run and inspect this example before implementing your own functions. It demonstrates the supplied infrastructure.

In [ ]:
from maintenance_course.state import StateStore
import tempfile
with tempfile.TemporaryDirectory() as folder:
    store = StateStore(Path(folder)/"state.db")
    report = run_case(fixed, case_by_id("D01"))["report"]
    try:
        store.apply_simulated("demo", report)
    except PermissionError as error:
        print("Expected rejection:", error)
    store.approve("demo", report, "human demonstration reviewer")
    print(store.apply_simulated("demo", report))
    store.close()
    store = StateStore(Path(folder)/"state.db")
    print(store.apply_simulated("demo", report))
    print({"recorded_actions":store.action_count()})
    store.close()
print("Approval was a separate controller action, never a model tool.")

## Your implementation

Implement `validate_plan(plan)` and `execute_plan(plan, registry, store, session, max_retries=1)` in `student_work/week06.py`. Use `StateStore` for persistence. Add a planner wrapper in your notebook or source that allows at most one plan revision.

Set `RUN_STUDENT` to `True` after completing the named functions. The default leaves unfinished functions unexecuted; it does not mark the assignment complete.

In [ ]:
import student_work.week06 as student
RUN_STUDENT = False
if RUN_STUDENT:
    result = student.validate_plan({"steps":[{"id":"inspect","name":"diagnose_csv","arguments":{"fixture_id":"missing-date"},"depends_on":[]}]})
    print(result)
else:
    print("Student implementation has not been evaluated. Complete the lab, then enable this cell.")

## Required experiment

1. **Define and reject plans.** Use one to five steps with unique IDs, permitted tool arguments, and dependencies on earlier steps. Validate the complete plan before execution. Demonstrate rejection of a cycle, forward dependency, duplicate ID, unknown tool, malformed arguments, and six-step plan.
2. **Generate and execute a plan.** Request a plan through `model.complete("plan", payload)`, validate it, then execute only ready steps. Save the plan digest and completed results after each step. Allow at most one retry for a timeout and no permission bypass. Compare a live plan or designated recorded plan with the fixed baseline on at least five development cases.
3. **Demonstrate recovery.** Interrupt after one completed step, close the state store, create new registry and store instances, and resume the same session. Verify that completed work is restored rather than executed again and that its evidence is available to the final recommendation.
4. **Handle a changed or failed plan.** Inject a persistent timeout and show termination after the retry bound. Permit at most one replan using the failure observation. Give a revised plan a new session identity. Reject a checkpoint when the case or plan digest does not match.
5. **Demonstrate approval and idempotency.** Create a report JSON file. Use the separate `approve` and `apply` commands or a human-controlled notebook section. Reject application before approval and after any payload change. Record the exact approved payload, restart the database connection, and repeat the request; the action count must remain one.
6. **Connect a peer contribution and the framework.** Adopt one attributed test or fixture from a peer repository, preserve its source commit, and show its effect on your validation. Run the supplied CrewAI Flow demonstration, then map its steps to your durable executor and explain which persistence and approval properties you implemented outside the simple demonstration.

## Results and analysis

Add your executed experiment cells and case-level table here. State the case set, variant, mode, model, success dimensions, resource measurements, and exclusions. Trace a failure to its cause and explain the resulting architecture decision. Export selected evidence into the submission folder.

## Research, interaction, and disclosure

Link the primary-source research note, discussion and peer response, and AI-use disclosure. Check every required artifact against the weekly rubric.